In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("./data/statutes_data/Central_Acts.csv", index_col=0)
df.insert(0, "State Name", "India")
df.head(15)

,State Name,Name of statute,Section Number,Section Title,Section Text
0,India,"The Essential Commodities Act, 1955",1.,Short title and extent.,(1) This Act may be called the Essential Commo...
1,India,"The Essential Commodities Act, 1955",2.,Definitions.,"In this Act, unless the context otherwise requ..."
2,India,"The Essential Commodities Act, 1955",2A.,"Essential commodities declaration, etc.","1[2A. Essential commodities declaration, etc.-..."
3,India,"The Essential Commodities Act, 1955",3.,"Powers to control production, supply, distribu...",(1) If\nthe Central Government is of opinion t...
4,India,"The Essential Commodities Act, 1955",4.,"Imposition of duties on State Governments, etc.",An order made under section 3 may\nconfer powe...
5,India,"The Essential Commodities Act, 1955",5.,Delegation of powers.,"The Central Government may, by notified order,..."
6,India,"The Essential Commodities Act, 1955",6.,Effect of orders inconsistent with other enact...,Any order made under section 3 shall\nhave eff...
7,India,"The Essential Commodities Act, 1955",6A.,Confiscation of essential commodity.,1[6A. Confiscation of essential commodity.--2[...
8,India,"The Essential Commodities Act, 1955",6B.,Issue of show cause notice before confiscation...,1[6B. Issue of show cause notice before confis...
9,India,"The Essential Commodities Act, 1955",6C.,Appeal.,1[6C. Appeal.--(1) Any person aggrieved by an ...


In [4]:
df2 = pd.read_csv("./data/statutes_data/State_Acts.csv", index_col=0)
df2.rename(columns={"Name of Statute": "Name of statute"}, inplace=True)
df2.head(15)

,State Name,Name of statute,Section Number,Section Title,Section Text
0,Andaman and Nicobar Islands,"The inland vessels act, 1917",1.,Short title and extent.,(1) This Act may be called the1[Inland Vessels...
1,Andaman and Nicobar Islands,"The inland vessels act, 1917",2.,Definitions.,"1[1] In this Act, unless there is anything rep..."
2,Andaman and Nicobar Islands,"The inland vessels act, 1917",3.,Inland mechanically propelled vessel not to pr...,1[3. Inland mechanically propelled vessel not ...
3,Andaman and Nicobar Islands,"The inland vessels act, 1917",4.,Appointment of surveyors and places of survey.,"(1) The State Government may, by notification ..."
4,Andaman and Nicobar Islands,"The inland vessels act, 1917",5.,Powers of surveyors.,"(1) For the purposes of a survey, the surveyor..."
5,Andaman and Nicobar Islands,"The inland vessels act, 1917",6.,Fees in respect of surveys.,"Before a survey is commenced, the owner or mas..."
6,Andaman and Nicobar Islands,"The inland vessels act, 1917",7.,Declaration of surveyor.,When the survey of a1[mechanically propelled v...
7,Andaman and Nicobar Islands,"The inland vessels act, 1917",8.,Sending of declaration by owner or master to S...,(1) The owner or master of a1[mechanically pro...
8,Andaman and Nicobar Islands,"The inland vessels act, 1917",9.,Power for State Government to grant or authori...,"(1) The State Government shall, if satisfied t..."
9,Andaman and Nicobar Islands,"The inland vessels act, 1917",9A.,Temporary permit.,"1The surveyor who conducted the survey may, wi..."


In [5]:
merged_df = pd.concat([df, df2], ignore_index=True)
merged_df.head()

,State Name,Name of statute,Section Number,Section Title,Section Text
0,India,"The Essential Commodities Act, 1955",1.,Short title and extent.,(1) This Act may be called the Essential Commo...
1,India,"The Essential Commodities Act, 1955",2.,Definitions.,"In this Act, unless the context otherwise requ..."
2,India,"The Essential Commodities Act, 1955",2A.,"Essential commodities declaration, etc.","1[2A. Essential commodities declaration, etc.-..."
3,India,"The Essential Commodities Act, 1955",3.,"Powers to control production, supply, distribu...",(1) If\nthe Central Government is of opinion t...
4,India,"The Essential Commodities Act, 1955",4.,"Imposition of duties on State Governments, etc.",An order made under section 3 may\nconfer powe...


In [6]:
print(len(merged_df), len(df), len(df2))

186338 36731 149607


In [7]:
import re

act_2_id = dict()
id_2_text = dict()
id = 0
for idx, row in merged_df.iterrows():
    # print(row)
    if isinstance(row['Name of statute'], float):
        continue
    act_name = row['Name of statute'].replace('_', ' ')
    act_name = act_name.capitalize()
    # Replace all non-alphanumeric characters with underscore
    section_name = re.sub(r'[^A-Za-z0-9]', '_', str(row['Section Number']))
    section_name = section_name.lower()  
    # Separate alphabets and numbers with underscore
    section_name = re.sub(r'([A-Za-z])([0-9])', r'\1_\2', section_name)
    section_name = re.sub(r'([0-9])([A-Za-z])', r'\1_\2', section_name)
    # Replace multiple underscores with single underscore
    section_name = re.sub(r'_+', '_', section_name)
    # Remove any leading/trailing underscores
    section_name = section_name.strip('_')
    # Remove '0' if it is at the start or end, bounded by underscore and word boundary
    section_name = re.sub(r'(^_?0\b|(?<=_)0\b|^0_|\b0_?$)', '', section_name)
    # Remove any leftover multiple underscores and strip again
    section_name = re.sub(r'_+', '_', section_name).strip('_')
    section_title = row['Section Title']
    section_text = row['Section Text']
    if(act_name not in act_2_id):
        act_2_id[act_name] = id
        id += 1
    act_id = act_2_id[act_name]
    if(act_id not in id_2_text):
        id_2_text[act_id] = {
            "Name": act_name,
            "Sections":{
                section_name: section_title + ": " + section_text
            }
        }
    else:
        id_2_text[act_id]["Sections"][section_name] = section_title + ": " + section_text

In [8]:
import json

with open("./data/statutes_data/act_2_id.json", "w", encoding="utf-8") as f:
    json.dump(act_2_id, f, ensure_ascii=False, indent=2)

with open("./data/statutes_data/id_2_text.json", "w", encoding="utf-8") as f:
    json.dump(id_2_text, f, ensure_ascii=False, indent=2)

In [9]:
import json
import random
print(json.dumps(id_2_text[random.choice(list(id_2_text.keys()))], indent=2, ensure_ascii=False))

{
  "Name": "The rajasthan home guards act, 1963",
  "Sections": {
    "1": "Short title, extent and commencement-: (1) This Act may be called the Rajasthan Home Guards Act, 1963. (2) It extends to the whole of the State of Rajasthan. (3) It shall come into force at once.",
    "3": "Appointment of the members -: (1) Subject to the approval of the Commandant General, the Commandant may appoint as members of the Home Guards such number of persons, who are fit and willing to serve as may from time to time be determined by the State Government, and may appoint any such members to any office of Command in the Home Guards. (2) Notwithstanding anything contained in sub-section (I) the Commandant General may, subject to the approval of the State Government appoint any such member to any post under his immediate control. 4 (1) The Commandant may at any time call out a member of Home Guards for training or to discharge any of the functions or duties assigned to the Home Guards in accordance wit

In [10]:
id_2_text[act_2_id["The Right to Information Act, 2005".capitalize()]]['Sections']['2']

'Definitions.—: (1) In these rules, unless the context otherwise requires, - (a) “Act” means the Right to Information Act, 2005 (22 of 2005); (b) “Central Information Commission” shall have the same meaning assigned to it under clause (b) of section 2 of the Act ; ¹Hkkx II µ[k.M 3 (i) º Hkkjr dk jkti=k % vlk/kj.k 5 (c) “Chief Information Commissioner” and “Information Commissioner” shall have the same meaning assigned to it under clause (d) of section 2 of the Act; (d) “State Chief Information Commissioner” and “State Information Commissioner” shall have the same meaning assigned to it under clause (l) of section 2 of the Act; (e) “State Information Commission” shall have the same meaning assigned to it under clause (k) of section 2 of the Act. (2) The words and expressions used and not defined under these rules, but defined in the Act shall have the same meaning as respectively assigned to them in the Act.'

In [56]:
# pip install sentence-transformers numpy tqdm

import numpy as np
from sentence_transformers import SentenceTransformer, util
from tqdm import tqdm

# # --- Step 1: Load Act Names ---
with open('./data/statutes_data/act_2_id.json', encoding='utf-8') as f:
    act_2_id = json.load(f)
    act_names = [name for name in act_2_id.keys()]

# # Optionally deduplicate names:
# act_names = list(dict.fromkeys(act_names))
# act_names = list(act_2_id.keys())
# --- Step 2: Normalize Helper ---
import re
def normalize(text):
    text = text.lower()
    text = re.sub(r'[^\w\s]', '', text)  # remove punctuation
    text = re.sub(r'\s+', ' ', text)     # standardize whitespace
    return text.strip()

normed_act_names = [normalize(name) for name in act_names]

# --- Step 3: Generate and Store Embeddings ---
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')  # because some act names are in Hindi and English

print("Generating embeddings for all acts, please wait...")
act_embeddings = model.encode(normed_act_names, convert_to_numpy=True, show_progress_bar=True, normalize_embeddings=True)



Generating embeddings for all acts, please wait...


Batches: 100%|██████████| 104/104 [00:00<00:00, 116.09it/s]


In [57]:
# --- Step 4: Save Embeddings and mappings ---
# act_names: list of canonical names
# act_embeddings: numpy array, shape (n_acts, embed_dim)

# Save embeddings
np.save("act_embeddings.npy", act_embeddings)

# Save mapping
with open("act_names.json", "w", encoding="utf-8") as f:
    json.dump(act_names, f, ensure_ascii=False, indent=2)


In [58]:
# --- Step 5: Load Embeddings and Mappings (if needed) ---
import numpy as np
import json

# Load embeddings
act_embeddings = np.load("act_embeddings.npy")

# Load mapping
with open("act_names.json", "r", encoding="utf-8") as f:
    act_names = json.load(f)

# --- Step 6: Function to Match User Query ---
def match_statute_name(user_input, act_embeddings, act_names, return_top_k=3, score_threshold=0.7):
    """Return top-k best matches and their similarity scores"""
    norm_query = normalize(user_input)
    query_emb = model.encode([norm_query], convert_to_numpy=True, normalize_embeddings=True)

    scores = np.dot(act_embeddings, query_emb[0])
    ranked_idx = np.argsort(scores)[::-1]  # Descending order

    # Filter by threshold and return top K
    results = []
    for idx in ranked_idx[:return_top_k]:
        score = scores[idx]
        if score < score_threshold:
            continue
        results.append({'act_name': act_names[idx], 'score': float(score)})
    return results


# Example user query
user_statute_query = "Arms Act"
matches = match_statute_name(user_statute_query, act_embeddings, act_names, return_top_k=3, score_threshold=0.6)
print("User input:", user_statute_query)
for m in matches:
    print(f"Closest match: {m['act_name']} (similarity: {m['score']:.3f})")


User input: Arms Act
Closest match: The arms act, 1959 (similarity: 0.723)
Closest match: UAP Act (similarity: 0.639)


In [59]:
import re

class StatuteExtractor:
    def __init__(self, act_names=None, common_abbreviations=None):
        # Compile possible act names (for strict matching), or use a generic fallback if none provided
        possible_act_names = []
        if act_names:
            for act in act_names:
                cleaned_act = re.sub(r'^\bThe\b\s+', '', act, flags=re.IGNORECASE)
                cleaned_act = re.sub(r',?\s+\d{4}$', '', cleaned_act)
                possible_act_names.append(cleaned_act)
            possible_act_names += act_names
        if common_abbreviations:
            possible_act_names += common_abbreviations
        # Remove empties and dedupe
        possible_act_names = [x for x in set(possible_act_names) if x]
        if possible_act_names:
            law_list_regex = r'(?:' + '|'.join(sorted(map(re.escape, possible_act_names), key=len, reverse=True)) + r')'
        else:
            # fallback to generic match for "Act", "Code", "Law", "Rules", "Regulations" etc.
            law_list_regex = r'[A-Za-z].*?(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines|IPC|CrPC|CPC)'
        # Pattern
        self.pattern = re.compile(
            r'''
            \b
            (?:(?:[Ss]ection|[Aa]rticle|[Ss]chedule)s?|[Uu]/[Ss])            # “Section(s)” or “Article(s)” or “u/s”
            \W+                                                  # separator (spaces, hyphens…)
            (?P<sections>                                        # one or more section‑number tokens
                (?:
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )
                (?:
                    \s*(?:,\s*|\s+and\s+)
                    (?:[Ss]ection|[Aa]rticle|[Ss]chedule\s*)?
                    \d+(?:\s*\(([A-Za-z]{1,2}|[0-9]{1,3})\)|[-_]([A-Za-z]{1,2}|[0-9]{1,3}))?
                )*
            )
            \s+
            ,?\s*
            (?:of\s+)?                                          # optional “of”
            (?:the\s+)?                                         # optional “the”
            (?P<law_name>
                (?:Code\s+(?:of|on|for|to)(?:\s+[A-Z][a-zA-Z]*)+)
              |
                (?:[A-Z][a-zA-Z]*(?:\s+[A-Z][a-zA-Z]*)*\s+(?:Act|Code|Law|Regulation|Rules|Ordinance|Bill|Amendment|Notification|Order|Guidelines))
              |
                ''' + law_list_regex + r'''
              |
                (?:[A-Za-z]{1,2}\.)+         # one or more “XX.” segments
                (?:[A-Za-z]{1,2}\.?)?        # optional final "XX" with optional dot
            )
            \b
            ''',
            flags=re.IGNORECASE | re.VERBOSE
        )

    def normalize_section_name(self, section):
        section_name = section.lower()  # Normalize to lowercase
        section_name = re.sub(r'[^A-Za-z0-9]', '_', str(section_name))
        section_name = re.sub(r'([A-Za-z])([0-9])', r'\1_\2', section_name)
        section_name = re.sub(r'([0-9])([A-Za-z])', r'\1_\2', section_name)
        section_name = re.sub(r'_+', '_', section_name)
        section_name = section_name.strip('_')
        section_name = re.sub(r'(^_?0\b|(?<=_)0\b|^0_|\b0_?$)', '', section_name)
        section_name = re.sub(r'_+', '_', section_name).strip('_')
        return section_name

    def extract(self, text):
        """Returns a list of {'section': ..., 'act': ...} elements."""
        matches = self.pattern.finditer(text)
        results = []
        seen = set()
        for match in matches:
            sections_chunk = match.group('sections')
            law_name = match.group('law_name').strip()
            raw_sections = re.split(r'(?:,|\band\b)', sections_chunk)
            for sect in raw_sections:
                sect = sect.strip()
                if sect:
                    sect = re.sub(r'^[Ss]ection\s*', '', sect)
                    sect = re.sub(r'^[Aa]rticle\s*', '', sect)
                    sect = self.normalize_section_name(sect)
                    name = f"section_{sect}_of_{law_name}"
                    if name in seen:
                        continue
                    seen.add(name)
                    results.append({'section': sect, 'act': law_name})
        return results

In [17]:
common_abbreviations = [
    'IPC', 'CrPC', 'CPC', 'IT Act', 'RTI Act', 'POSCO', 'SC/ST Act', 'PoA', 'FCRA', 'FEMA', 'PMLA', 'NDPS', 'HMA', 'DV Act',
    'DVC', 'NIA', 'RERA', 'DRT', 'CGST Act', 'SEBI Act', 'MVA', 'ESMA', 'POTA', 'PoTA', 'COFEPOSA', 'CoFEPoSA', 'CAT', 'PSA',
    'UAPA', 'NSA', 'MCOCA', 'SARFAESI', 'SARFAESI Act', 'JJ Act', 'SHWW Act', 'POSH Act', 'SHWW', 'POSH', 'PoSH', 'FEOA', 'GI Act',
    'PCA', 'NDPS', 'NDPS Act', 'Dowry Act', 'Dowry Proh. Act', 'IEA', 'UPPRA', 'UPPCA', 'Cow Slaughter Act', 'UPPDA', 'UPPFA', 'Goonda Act', 'Gangsters Act',
]
extractor = StatuteExtractor(common_abbreviations=common_abbreviations)

In [18]:
count = dict()

In [ ]:
with open("data/final_cleaned_alt.json", "r", encoding="utf-8") as f:
    case_data = json.load(f)

for item in tqdm(case_data, desc="counting statutes"):
    case = item.get('case', '')
    extracted_statutes = extractor.extract(case)
    for statute in extracted_statutes:
        if statute['act'].lower() in ['ipc', 'crpc', 'i.p.c.', 'cr.p.c.','cr.pc.', 'i.p.c', 'cr.p.c', 'cr.pc', 'indian penal code', 'code of criminal procedure', 'criminal procedure code']:
            continue
        count[statute['act']] = count.get(statute['act'], 0) + 1

counting statutes: 100%|██████████| 208292/208292 [00:22<00:00, 9395.82it/s] 


In [60]:
# Example user query
user_statute_query = "PoCSO Act"
matches = match_statute_name(user_statute_query, act_embeddings, act_names, return_top_k=3, score_threshold=0.6)
print("User input:", user_statute_query)
for m in matches:
    print(f"Closest match: {m['act_name']} (similarity: {m['score']:.3f})")

User input: PoCSO Act
Closest match: POCSO Act, 2012 (similarity: 0.785)
Closest match: PIT Act (similarity: 0.742)
Closest match: UAP Act (similarity: 0.691)


In [22]:
import json
count = dict(sorted(count.items(), key=lambda x: x[1], reverse=True))
print(json.dumps(count, indent=4, ensure_ascii=False))

{
    "Arms Act": 11338,
    "POCSO Act": 6322,
    "Protection of Children from Sexual Offences Act": 4725,
    "Abkari Act": 4566,
    "Dowry Prohibition Act": 3114,
    "Indian Forest Act": 2969,
    "Kerala Abkari Act": 2874,
    "IT Act": 2814,
    "NDPS Act": 2380,
    "Explosive Substance Act": 1987,
    "CLA Act": 1720,
    "Excise Act": 1529,
    "N.D.P.S": 1405,
    "D.P": 1361,
    "Kerala Protection of River Banks and Regulation of Removal of Sand Act": 1358,
    "Prevention of Corruption Act": 1259,
    "C.L.A": 1169,
    "Criminal Law Amendment Act": 1042,
    "Information Technology Act": 994,
    "Wild Life Protection Act": 979,
    "Maharashtra Police Act": 770,
    "U.P": 767,
    "Essential Commodities Act": 759,
    "Explosive Substances Act": 755,
    "Narcotic Drugs and Psychotropic Substances Act": 725,
    "Coal Mines Act": 685,
    "Kerala Money Lenders Act": 591,
    "UAP Act": 562,
    "MV Act": 555,
    "Bombay Police Act": 542,
    "Indian Arms Act": 477,
 